# Incremental indexing — adds, deletes, tombstones, re-embeds

A real index isn't built once. Documents arrive, get edited, get deleted; embedding models change. This notebook walks through the five mutations every production vector index has to handle:

1. **add** — append a new doc and its vector.
2. **delete** — mark a doc as gone, but don't pay for a full rebuild (a.k.a. **tombstoning**).
3. **update** — replace an existing doc's vector in place.
4. **rebuild** — periodic compaction that drops tombstones.
5. **re-embed** — full or partial recomputation when the embedding model changes.

We use a hand-written flat cosine index so the mutations are obvious, then measure recall before and after each operation.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


In [ ]:
import numpy as np
from shared.embedders import hash_embed
from shared.loaders import load_corpus, load_golden_qa

DIMS = 256
SEED_OLD = 0
SEED_NEW = 7
DOCS = load_corpus()
QA = [q for q in load_golden_qa() if q.source_ids]
print(f'{len(DOCS)} docs, {len(QA)} questions')

## A flat index with `add`, `delete`, `replace`

`delete` is **tombstoned** — we add to `deleted` and filter at query time. This mirrors what FAISS, Qdrant, and pgvector do; physical compaction happens later in a rebuild.

In [ ]:
class FlatIndex:
    def __init__(self):
        self.vecs, self.ids, self.deleted = [], [], set()
    def add(self, doc_id, vec):
        v = vec / (np.linalg.norm(vec) + 1e-9)
        self.vecs.append(v); self.ids.append(doc_id)
    def delete(self, doc_id):
        i = self.ids.index(doc_id); self.deleted.add(i)
    def replace(self, doc_id, vec):
        i = self.ids.index(doc_id)
        self.vecs[i] = vec / (np.linalg.norm(vec) + 1e-9)
    def search(self, qv, k=3):
        qv = qv / (np.linalg.norm(qv) + 1e-9)
        scores = np.stack(self.vecs) @ qv
        out = []
        for i in np.argsort(-scores):
            if int(i) in self.deleted: continue
            out.append(self.ids[int(i)])
            if len(out) == k: break
        return out

def embed(text, seed=SEED_OLD): return hash_embed([text], dims=DIMS, seed=seed)[0]

def recall(idx, qa, embed_q, k=3):
    hits = sum(1 for q in qa if set(idx.search(embed_q(q.question), k)) & set(q.source_ids))
    return round(hits / len(qa), 4)

## Step 1 — baseline

In [ ]:
idx = FlatIndex()
for d in DOCS:
    idx.add(d.arxiv_id, embed(d.title + '. ' + d.abstract))
print('baseline recall@3 =', recall(idx, QA, embed))

## Step 2 — tombstone delete

Drop two papers. Filter the golden Q&A to those whose answers no longer have any source in the live index.

In [ ]:
to_delete = [DOCS[0].arxiv_id, DOCS[2].arxiv_id]
qa_live = [q for q in QA if not set(q.source_ids).issubset(set(to_delete))]
for did in to_delete: idx.delete(did)
print(f'deleted {to_delete}; live docs = {len(idx.vecs) - len(idx.deleted)}')
print('post-delete recall@3 =', recall(idx, qa_live, embed))

## Step 3 — add new docs

In [ ]:
extras = [
    ('synth-101', 'Reproducibility study of speculative decoding under fp8 quantisation.'),
    ('synth-102', 'Survey of retrieval failure modes in legal document QA systems.'),
]
for did, text in extras:
    idx.add(did, embed(text))
print(f'live docs = {len(idx.vecs) - len(idx.deleted)}')
print('post-add recall@3 =', recall(idx, qa_live, embed))

## Step 4 — partial re-embed (the cliff)

Re-embed half the corpus with a new seed (= a new embedding model). Queries still use the old embedder. Recall craters — vectors from different embedders live in different geometries and **cannot be compared cosine-wise**.

In [ ]:
half = len(DOCS) // 2
for d in DOCS[:half]:
    idx.replace(d.arxiv_id, embed(d.title + '. ' + d.abstract, seed=SEED_NEW))
print('mixed-embedder recall@3 (old queries) =', recall(idx, qa_live, lambda t: embed(t, SEED_OLD)))
print('mixed-embedder recall@3 (new queries) =', recall(idx, qa_live, lambda t: embed(t, SEED_NEW)))

## Takeaways

* **Tombstoning** is free at delete time; compaction is the periodic rebuild that reclaims space.
* **Adds** are O(1) for flat / O(log N) for HNSW. No retraining needed.
* **Mixed embedders break everything** — when you change embedding models, you MUST re-embed the entire corpus *and* the queries. There's no incremental migration path that preserves recall.
* In production: keep an **embedder version** field on every vector; reject queries embedded with a different version; run a background re-embed job before flipping the flag.